# 05 — Entrenar TrajectoryLSTM ④

Entrena el LSTM de trayectoria + clasificación táctica. Mientras no haya trayectorias
anotadas reales, usamos trayectorias sintéticas (`synthetic_trajectory`) para validar
que el pipeline funciona y reportarlo como baseline en el report.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content
!rm -rf volley-ai
!git clone https://github.com/Angelote567/DeepLearning.git volley-ai
%cd volley-ai
!pip install -q torch torchvision tqdm Pillow
import os
if os.path.exists('weights') and not os.path.islink('weights'):
    if not os.listdir('weights') or os.listdir('weights') == ['.gitkeep']:
        import shutil; shutil.rmtree('weights')
if not os.path.exists('weights'):
    os.symlink('/content/drive/MyDrive/volley-ai/weights', 'weights')

In [ ]:
import json, os, random
from src.data.sequence_dataset import synthetic_trajectory

samples = synthetic_trajectory(num=2000)
random.shuffle(samples)
splits = {'train': samples[:1600], 'valid': samples[1600:1800], 'test': samples[1800:]}
for split, items in splits.items():
    out_dir = f'data/trajectories/{split}'
    os.makedirs(out_dir, exist_ok=True)
    for i, s in enumerate(items):
        with open(f'{out_dir}/{i:05d}.json', 'w') as f:
            json.dump(s, f)
print('OK trayectorias sintéticas generadas')

In [ ]:
import torch
from torch.utils.data import DataLoader
from src.data.sequence_dataset import TrajectoryDataset
from src.models.lstm import TrajectoryLSTM, trajectory_loss

device = 'cuda' if torch.cuda.is_available() else 'cpu'
train_ds = TrajectoryDataset('data/trajectories', split='train')
val_ds = TrajectoryDataset('data/trajectories', split='valid')
train_dl = DataLoader(train_ds, batch_size=64, shuffle=True)
val_dl = DataLoader(val_ds, batch_size=64)

model = TrajectoryLSTM().to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
best = float('inf')
for epoch in range(40):
    model.train()
    tl = 0
    for x, y, tac in train_dl:
        x, y, tac = x.to(device), y.to(device), tac.to(device)
        pred_xy, pred_tac = model(x)
        loss, _ = trajectory_loss(pred_xy, y, pred_tac, tac)
        opt.zero_grad(); loss.backward(); opt.step()
        tl += loss.item()
    model.eval()
    vl = 0
    with torch.no_grad():
        for x, y, tac in val_dl:
            x, y, tac = x.to(device), y.to(device), tac.to(device)
            pred_xy, pred_tac = model(x)
            loss, _ = trajectory_loss(pred_xy, y, pred_tac, tac)
            vl += loss.item()
    vl /= len(val_dl); tl /= len(train_dl)
    print(f'epoch {epoch+1} train {tl:.4f} val {vl:.4f}')
    if vl < best:
        best = vl
        torch.save({'model': model.state_dict()}, 'weights/trajectory_lstm.pt')
print('OK')